# Generics for Reusable ADTs

You have been using generics since Chapter 8: `List<T>` and `Dictionary<TKey, TValue>`. In an ADT, generics let one contract and one implementation work with many element types while preserving compile-time type checking.

For the caller obligations of the ref swap exercise, review {ref}`Chapter 5 parameter passing <gap-return-values-out-and-ref>`.


```{contents}
:local:
:depth: 2
```

```{index} generic type
```



```{index} Why Generics Exist
```

## Why Generics Exist




Without generics, a stack that holds `int` would need to be a different class from a stack that holds `string`, or it would have to store everything as `object` and lose type safety. Generics let one implementation work for many types:

```csharp
// Non-generic — stores object, loses type safety
var list = new ArrayList();
list.Add(1);
list.Add("oops");  // compiles — bug at runtime

// Generic — type-safe at compile time
var nums = new List<int>();
nums.Add(1);
// nums.Add("oops");  // compile error
```

For related explanation, see the further reading below.[^development-reading]


```{index} Reuse an Algorithm Without Erasing Its Types
```



```{index} Reuse an Algorithm Without Erasing Its Types
```

## Reuse an Algorithm Without Erasing Its Types



An undo history of product codes and a history of numeric adjustments can follow the same push/pop rules. Writing two unrelated implementations repeats the same invariant and creates two places to repair a defect. A generic type makes the element type a parameter while leaving the behavioral algorithm shared.

Read `History<T>` as a type definition with an element-type placeholder. Read `History<string>` as one constructed type whose element operations must accept strings. The placeholder is not a variable containing an element and is not a request to decide the type afresh on every assignment. A History<int> and a History<string> are different constructed types even though their operation names are the same.

The general class pattern is `class Container<T> { /* members that use T */ }`. A generic method has its own type parameter list, as in `static T Identity<T>(T value)`. These are declaration patterns, not complete programs. In concrete use the compiler checks compatible argument/result types. That lets a bad element insertion be rejected before a client discovers an incompatible value during a later cast.




```{index} State Which Operations the Generic Code Needs
```



```{index} State Which Operations the Generic Code Needs
```

## State Which Operations the Generic Code Needs



An unconstrained T supports being stored, passed and returned. It does not promise a Price property or a meaningful greater-than operation. If the algorithm needs ordering, supply an ordering policy or an appropriate constraint. If it needs key identity, supply an equality/hash policy with its consistency requirements. Ordering and equality are related concepts but different contracts.

| Algorithm purpose | Information needed beyond storage |
| --- | --- |
| Return the most recently stored item | No ordering comparison |
| Sort values for a report | A consistent ordering policy |
| Find a key in a hash table | Equality and compatible hashing |
| Call an element-specific capability | An explicit capability constraint |

A comparer can say that two invoices have equal totals even when they are distinct invoices. A key-equality policy might instead use invoice ID. Choosing a generic collection does not choose those business meanings automatically. Avoid solving a missing requirement by converting every value to text and comparing that text; it can change numeric order and lose distinctions.

Guided design: create a generic history interface for values that never need to be compared. Would requiring all element types to implement an ordering interface help? It would unnecessarily reject useful element types. Add constraints because an operation needs them, not because generics seem advanced. The short lambda/query forms later in this lesson are a bridge for reading policy arguments; the fundamental generic contract does not depend on mastering LINQ.


```{index} Generic Methods
```



```{index} Generic Methods
```

## Generic Methods



In [1]:
using System;
using System.Collections.Generic;
using System.Linq;
// A generic method works for any type T
static T[] Repeat<T>(T value, int count)
{
    var arr = new T[count];
    for (int i = 0; i < count; i++) arr[i] = value;
    return arr;
}

int[]    ints    = Repeat(0, 5);       // [0, 0, 0, 0, 0]
string[] strings = Repeat("hi", 3);   // ["hi", "hi", "hi"]

Console.WriteLine(string.Join(", ", ints));     // 0, 0, 0, 0, 0
Console.WriteLine(string.Join(", ", strings));  // hi, hi, hi

0, 0, 0, 0, 0
hi, hi, hi


```{index} generic class
```



```{index} Generic Classes
```

## Generic Classes



In [1]:
using System;
using System.Collections.Generic;
using System.Linq;
var score = new Cell<int>(95);
var name = new Cell<string>("Ada");

score.Set(98);
Console.WriteLine(score.Get());
Console.WriteLine(name.Get());

// A minimal generic cell ADT
class Cell<T>
{
    private T value;

    public Cell(T initialValue) => value = initialValue;
    public T Get() => value;
    public void Set(T newValue) => value = newValue;
}

98
Ada


```{index} type constraint
```



```{index} Type Constraints
```

## Type Constraints




Use `where T : ...` to restrict what types `T` can be. Constraints are useful when an ADT operation needs a capability that not every type has.

| Constraint | Meaning |
|---|---|
| `where T : class` | T must be a reference type |
| `where T : struct` | T must be a value type |
| `where T : new()` | T must have a parameterless constructor |
| `where T : ISomeInterface` | T must implement the interface |
| `where T : BaseClass` | T must inherit from `BaseClass` |

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;
// Constraint: T must implement IComparable<T> so we can compare
static T Max<T>(T a, T b) where T : IComparable<T>
    => a.CompareTo(b) >= 0 ? a : b;

Console.WriteLine(Max(3, 7));          // 7
Console.WriteLine(Max("apple", "banana")); // banana

7
banana


```{index} generic interface
```



```{index} Generic Interfaces
```

## Generic Interfaces




An ADT contract can be generic too. `IStack<T>` describes stack behavior for any element type `T`, while each implementation decides how to store those elements.

```csharp
interface IStack<T>
{
    int Count { get; }
    void Push(T item);
    T Pop();
    T Peek();
}
```

This is the same idea behind library interfaces such as `IEnumerable<T>`, `ICollection<T>`, and `IList<T>`: the interface names the behavior, and the type parameter names the element type.

```{index} Ordering and Equality Are Different Contracts
```

## Ordering and Equality Are Different Contracts




```{index} Ordering and Equality Are Different Contracts
```


An ordering comparer answers before, tied, or after with a negative value, zero, or a positive value. An equality comparer answers whether two values identify the same key. A price tie does not make two products the same product. Supply the same ordering policy to sorting and binary search; supply matching Equals and GetHashCode policies to hash collections. Equal keys must have equal hash codes, but unequal keys can collide. Never change a stored key in a way that changes its equality or hash value.


In [1]:
using System.Linq;
using System;
using System.Collections.Generic;
Invoice[] invoices = { new Invoice("A", 30m), new Invoice("B", 10m), new Invoice("C", 30m) };
Array.Sort(invoices, new AmountComparer());
foreach (Invoice invoice in invoices) Console.WriteLine($"{invoice.Id}: {invoice.Amount}");
class Invoice
{
    public string Id { get; }
    public decimal Amount { get; }
    public Invoice(string id, decimal amount) { Id = id; Amount = amount; }
}
class AmountComparer : IComparer<Invoice>
{
    public int Compare(Invoice? left, Invoice? right)
    {
        if (left is null || right is null) throw new ArgumentNullException();
        return left.Amount.CompareTo(right.Amount);
    }
}


B: 10
A: 30
C: 30


The named AmountComparer implements IComparer<Invoice>. Comparing Amount uses decimal.CompareTo, which avoids arithmetic subtraction and conversion. A comparer must be consistent and transitive. Array.Sort does not promise arrival order for tied values. A stable sorting algorithm preserves that order; Chapter 16 makes the distinction visible.


In [1]:
using System.Linq;
using System;
using System.Collections.Generic;
Dictionary<string, int> stock = new Dictionary<string, int>(StringComparer.OrdinalIgnoreCase);
stock.Add("P-100", 4);
Console.WriteLine(stock["p-100"]);
IEqualityComparer<string> identity = StringComparer.OrdinalIgnoreCase;
Console.WriteLine(identity.Equals("P-100", "p-100"));
Console.WriteLine(identity.GetHashCode("P-100") == identity.GetHashCode("p-100"));


4
True
True


```{index} Reading Short Expressions and Queries
```

## Reading Short Expressions and Queries




```{index} Reading Short Expressions and Queries
```


Later collection examples sometimes use compact C# forms. The expression x => x > 0 is a lambda: a function receiving x and returning the comparison result. A library method can call that function for each value. Where filters values, Select transforms them, OrderBy orders them, and ToArray materializes a separate array. These LINQ extension methods require using System.Linq. Prefer an explicit loop when it makes the algorithm or mutation clearer. Query pipelines can defer work until enumeration; ToArray executes the pipeline now.


In [1]:
using System.Collections.Generic;
using System;
using System.Linq;
int[] arrivals = { 30, -1, 10, 20 };
int[] report = arrivals.Where(value => value >= 0).OrderBy(value => value).Select(value => value * 2).ToArray();
Console.WriteLine(string.Join(", ", report));
Console.WriteLine(string.Join(", ", arrivals));


20, 40, 60
30, -1, 10, 20


An expression-bodied member such as public int Count => items.Count; is shorthand for a getter returning that expression. A target-typed new() uses the declared variable type, so List<int> values = new(); constructs a List<int>. Neither abbreviation changes the data-structure contract. Named classes and explicit constructors remain useful while learning a new model.


```{index} Practice
```

## Practice




1. Write a generic `static T First<T>(List<T> list)` that returns the first element or throws `InvalidOperationException` if the list is empty.
2. Write a generic `static void Swap<T>(ref T a, ref T b)` and test it with `int` and `string`.
3. Add a generic `IQueue<T>` interface with `Count`, `Enqueue`, `Dequeue`, and `Peek` members.
4. Explain why `IStack<T>` is a better ADT contract than separate `StringStack`, `IntStack`, and `BookStack` interfaces.

```{rubric} Footnotes
```


[^development-reading]: Further reading: [OpenDSA, Lists: Chapter Introduction](https://opendsa.org/OpenDSA/Books/Catalog/html/ListIntro.html). The worked business scenarios and guided reasoning here are original teaching examples.
